# Notebook 08 — Train Two Classifiers

## Purpose
This notebook performs **model training** for the top-performing classifiers  
identified in Notebook 07 using the normalized **25-dimensional Digital Image  
Processing (DIP) feature vectors**.

Rather than selecting a single model, this notebook trains **two classifiers**  
(e.g., RBF SVM and MLP) using their optimized hyperparameters and prepares them  
for final independent evaluation.

---

## Inputs
This notebook uses the normalized training dataset generated by:

- `06_Normalize_and_Prepare_Classifier_Input.ipynb`

Expected input files:

- `train_feature_vectors_normalized.csv`

The dataset contains:

- Metadata columns:
  - `filename`
  - `class_label`
  - `source_dataset`
  - `subset`
- **25 normalized DIP feature columns**

The classifier configurations are defined within this notebook (based on  
results from Notebook 07).

---

## Local Execution Assumptions
This notebook is designed to run within the **local GitHub project structure**  
or a compatible environment (e.g., Google Colab).

It assumes:

- the project repository is available locally or cloned at runtime
- `src/project_config.py` is accessible
- prior pipeline notebooks have generated the required normalized dataset
- required Python packages (NumPy, pandas, scikit-learn) are installed

---

## Important Note
All features have already been normalized using a scaler fit only on the  
training dataset in the previous notebook. No additional normalization  
should be applied here.

The independent test dataset exists, but it is **not used in this notebook**  
and remains reserved for final evaluation.

---

## Process Overview
This notebook trains and validates the selected classifiers using a consistent  
cross-validation framework. The workflow includes:

1. loading and validating the normalized training dataset  
2. separating metadata from feature columns  
3. defining classifier configurations based on prior selection results  
4. initializing classifiers with tuned hyperparameters  
5. performing stratified k-fold cross-validation  
6. evaluating models using multiple performance metrics  
7. comparing classifier performance  
8. training both classifiers on the full training dataset  
9. saving trained models and configuration summaries for downstream evaluation  

---

## Outputs
This notebook produces:

- cross-validation performance results for both classifiers:
  - `metadata/results/cross_validation_results.csv`

- saved model files:
  - `metadata/models/final_rbf_svm_model.pkl`
  - `metadata/models/final_mlp_model.pkl`

- saved JSON configuration files:
  - `metadata/models/best_rbf_svm_model_config.json`
  - `metadata/models/best_mlp_model_config.json`

These outputs are used in the final evaluation stage.

---

## Key Design Choice
This notebook performs **multi-model training**, not single-model selection.

Both top-performing classifiers from Notebook 07 are:

- trained using identical data and evaluation procedures  
- retained for final comparison on the independent test set  

This design:

- ensures consistency between selection and training stages  
- enables fair comparison between different model types  
- strengthens the validity of final evaluation results  

---

## Classifiers Trained
The classifiers trained in this notebook are:

- RBF Support Vector Machine (RBF SVM)  
- Multi-Layer Perceptron (MLP)  

Both models use hyperparameters determined during controlled tuning  
in Notebook 07.

---

## Scope Limitation
This notebook does **not** perform:

- classifier selection (performed in Notebook 07)  
- evaluation on the independent test dataset  

These steps are performed in subsequent notebooks.

---

## Cell-by-Cell Structure

### Cell 1
Import required libraries.

### Cell 2
Load the normalized training dataset and preview its contents.

### Cell 3
Validate dataset integrity and prepare feature matrix (`X_train`) and encoded target vector (`y_train`).

### Cell 4
Define selected classifier configurations.

### Cell 5
Run cross-validation for both selected classifiers.

### Cell 6
Summarize and compare classifier performance.

### Cell 7
Train both classifiers on the full training dataset.

### Cell 8
Save trained models to `.pkl` files in `metadata/models/`.

### Cell 9
Save cross-validation results to `metadata/results/` and per-model configuration files to `metadata/models/`.

In [1]:
# ============================================
# Startup (Environment + Verification)
# ============================================

import os
import sys
import json
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder

# -------------------------------------------------
# Clone repo into Colab runtime (if not already present)
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Make src/ importable
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import (
    TRAIN_NORMALIZED_FILENAME,
    NUM_FEATURES,
    RANDOM_SEED,
    METADATA_COLUMNS,
    AI_LABEL,
    REAL_LABEL,
    K_FOLDS,
    CV_SHUFFLE,
    CV_RANDOM_STATE,
)

# -------------------------------------------------
# Notebook 08 path overrides
# -------------------------------------------------
METADATA_ROOT = REPO_DIR / "metadata"
INPUT_CSV_DIR = METADATA_ROOT / "vectors"
MODELS_DIR = METADATA_ROOT / "models"
RESULTS_DIR = METADATA_ROOT / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Define required input paths
# -------------------------------------------------
TRAIN_FEATURE_VECTORS_NORMALIZED_CSV = INPUT_CSV_DIR / TRAIN_NORMALIZED_FILENAME

# -------------------------------------------------
# Define output paths
# -------------------------------------------------
FINAL_MLP_MODEL_PATH = MODELS_DIR / "final_mlp_model.pkl"
FINAL_RBF_SVM_MODEL_PATH = MODELS_DIR / "final_rbf_svm_model.pkl"

BEST_MLP_MODEL_CONFIG_PATH = MODELS_DIR / "best_mlp_model_config.json"
BEST_RBF_SVM_MODEL_CONFIG_PATH = MODELS_DIR / "best_rbf_svm_model_config.json"

CV_RESULTS_CSV = RESULTS_DIR / "cross_validation_results.csv"

# -------------------------------------------------
# Verify required input files
# -------------------------------------------------
print("Verifying required input files...\n")

missing_files = []

if not TRAIN_FEATURE_VECTORS_NORMALIZED_CSV.exists():
    missing_files.append(TRAIN_NORMALIZED_FILENAME)

if missing_files:
    raise FileNotFoundError(f"Missing required files: {missing_files}")

print("All required input files are present.")
print(f"Training data:    {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}")
print(f"Models directory: {MODELS_DIR}")
print(f"Results directory:{RESULTS_DIR}")


Cloning repository...
Verifying required input files...

All required input files are present.
Training data:    /content/dip-ai-image-detection/metadata/vectors/train_feature_vectors_normalized.csv
Models directory: /content/dip-ai-image-detection/metadata/models
Results directory:/content/dip-ai-image-detection/metadata/results


In [2]:
# ============================================================
# Cell 1: Import Required Libraries
# ============================================================

import os
import json
import pickle
import warnings
import time

import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")



Libraries imported successfully.


In [3]:
# ============================================================
# Cell 2: Load Normalized Training Data
# ============================================================

train_csv_path = INPUT_CSV_DIR / TRAIN_NORMALIZED_FILENAME

# -------------------------------------------------
# Load dataset
# -------------------------------------------------
df_train = pd.read_csv(train_csv_path)

print("Normalized training feature table loaded successfully.\n")

print(f"Training CSV: {train_csv_path}\n")

print(f"df_train shape: {df_train.shape}\n")

# -------------------------------------------------
# Preview data
# -------------------------------------------------
print("First 5 rows of training table:")
try:
    display(df_train.head())
except:
    print(df_train.head())



Normalized training feature table loaded successfully.

Training CSV: /content/dip-ai-image-detection/metadata/vectors/train_feature_vectors_normalized.csv

df_train shape: (14400, 29)

First 5 rows of training table:


,filename,class_label,source_dataset,subset,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,...,Patch Variance Std,Noise Residual Energy,Low Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,rl_imgn_002320.png,rl,ImageNet_1K_256,train,0.973214,1.208264,0.913526,0.776979,0.735101,-0.207859,...,1.019527,1.071297,-0.189219,0.417247,0.448644,0.439427,-0.549996,-0.084929,0.290945,-1.254194
1,rl_coco_001397.png,rl,MS_COCO_2017,train,-0.450913,-0.225225,0.641767,-0.488332,-0.447341,-0.246587,...,0.509586,-0.330229,-0.264290,0.144170,-1.327122,-1.325683,1.397134,0.856623,0.808074,0.071118
2,rl_imgn_001958.png,rl,ImageNet_1K_256,train,0.605933,0.723094,0.717432,0.610900,0.160581,-0.468100,...,0.235488,0.851489,-0.544033,0.805334,-0.343146,-0.332461,-0.549996,-0.217414,0.432281,-1.341491
3,rl_coco_000800.png,rl,MS_COCO_2017,train,0.021388,0.357107,0.528482,-0.077406,0.331237,1.075664,...,0.395378,0.188882,0.518813,-0.349898,1.963337,1.956027,-0.549996,-0.448685,-0.628062,-1.111831
4,ai_mj_002892.png,ai,Midjourney,train,-0.212459,-0.661686,-0.209270,0.288772,-0.061089,-0.293840,...,-0.373574,-0.281104,0.607047,-0.556607,-0.011009,-0.013482,-0.549996,-0.185376,-0.470299,0.325939


In [4]:
# ============================================================
# Cell 3: Validate Training Data and Prepare Features/Labels
# ============================================================

print("Performing sanity checks and preparing training data...\n")

# ------------------------------------------------------------
# Expected metadata columns
# ------------------------------------------------------------
expected_metadata_columns = METADATA_COLUMNS.copy()

# ------------------------------------------------------------
# Verify required metadata columns
# ------------------------------------------------------------
missing_train_metadata = [
    col for col in expected_metadata_columns
    if col not in df_train.columns
]

if missing_train_metadata:
    raise ValueError(
        f"Training table is missing metadata columns: {missing_train_metadata}"
    )

print("Required metadata columns verified.")

# ------------------------------------------------------------
# Verify feature columns
# ------------------------------------------------------------
feature_columns = [
    col for col in df_train.columns
    if col not in expected_metadata_columns
]

if len(feature_columns) != NUM_FEATURES:
    raise ValueError(
        f"Expected {NUM_FEATURES} feature columns, "
        f"but found {len(feature_columns)}."
    )

print(f"Feature count verified: {len(feature_columns)}")

# ------------------------------------------------------------
# Verify no missing values
# ------------------------------------------------------------
if df_train.isnull().sum().sum() != 0:
    raise ValueError("Training table contains missing values.")

print("No missing values detected.")

# ------------------------------------------------------------
# Separate features and labels
# ------------------------------------------------------------
X_train = df_train[feature_columns].copy()
y_train = df_train["class_label"].copy()

# ------------------------------------------------------------
# Verify label consistency
# ------------------------------------------------------------
expected_labels = {AI_LABEL, REAL_LABEL}
train_labels_found = set(y_train.unique())

if not train_labels_found.issubset(expected_labels):
    raise ValueError(
        f"Unexpected labels found in training set: {train_labels_found}"
    )

print(f"Label values verified: {sorted(expected_labels)}")

# ------------------------------------------------------------
# Encode labels
# ------------------------------------------------------------
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

print("\nLabel encoding mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label} -> {i}")

# ------------------------------------------------------------
# Display class distribution
# ------------------------------------------------------------
print("\nTraining class counts:")
print(y_train.value_counts())

# ------------------------------------------------------------
# Display matrix/vector shapes
# ------------------------------------------------------------
print("\nFeature matrix shape:")
print(f"X_train: {X_train.shape}")

print("\nLabel vector shapes:")
print(f"y_train:         {y_train.shape}")
print(f"y_train_encoded: {y_train_encoded.shape}")

print("\nSanity checks passed successfully.")



Performing sanity checks and preparing training data...

Required metadata columns verified.
Feature count verified: 25
No missing values detected.
Label values verified: ['ai', 'rl']

Label encoding mapping:
ai -> 0
rl -> 1

Training class counts:
class_label
rl    7200
ai    7200
Name: count, dtype: int64

Feature matrix shape:
X_train: (14400, 25)

Label vector shapes:
y_train:         (14400,)
y_train_encoded: (14400,)

Sanity checks passed successfully.


In [5]:
# ============================================================
# Cell 4: Define Selected Classifier Configurations
# ============================================================

print("Defining selected classifier configurations...\n")

classifier_configs = [
    {
        "classifier": "MLP",
        "params": {
            "hidden_layer_sizes": [64, 32],
            "alpha": 0.0001,
            "learning_rate_init": 0.001,
            "batch_size": 32
        }
    },
    {
        "classifier": "RBF SVM",
        "params": {
            "C": 100,
            "gamma": 0.01
        }
    }
]

print(f"Number of classifiers configured: {len(classifier_configs)}\n")

for i, config in enumerate(classifier_configs, 1):
    print(f"{i}. {config['classifier']}")
    print("   Parameters:")
    for param, value in config["params"].items():
        print(f"     {param} = {value}")
    print()



Defining selected classifier configurations...

Number of classifiers configured: 2

1. MLP
   Parameters:
     hidden_layer_sizes = [64, 32]
     alpha = 0.0001
     learning_rate_init = 0.001
     batch_size = 32

2. RBF SVM
   Parameters:
     C = 100
     gamma = 0.01



In [6]:
# ============================================================
# Cell 5: Cross-Validate Selected Classifiers
# ============================================================

cv = StratifiedKFold(
    n_splits=K_FOLDS,
    shuffle=CV_SHUFFLE,
    random_state=CV_RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
    "roc_auc": "roc_auc",
}

results = []

print("Evaluating selected classifiers...\n")
print(f"Number of classifiers: {len(classifier_configs)}")
print(f"Cross-validation folds: {K_FOLDS}\n")

total_start_time = time.time()

for i, config in enumerate(classifier_configs, start=1):
    model_start_time = time.time()

    classifier_name = config["classifier"]
    params = config["params"].copy()

    print(f"[{i}/{len(classifier_configs)}] {classifier_name}")

    # --------------------------------------------------------
    # Instantiate model
    # --------------------------------------------------------
    if classifier_name == "MLP":
        if "hidden_layer_sizes" in params:
            params["hidden_layer_sizes"] = tuple(params["hidden_layer_sizes"])
        params["max_iter"] = 300
        params["random_state"] = RANDOM_SEED
        model = MLPClassifier(**params)

    elif classifier_name == "RBF SVM":
        params["kernel"] = "rbf"
        params["probability"] = True
        params["random_state"] = RANDOM_SEED
        model = SVC(**params)

    else:
        raise ValueError(f"Unsupported classifier: {classifier_name}")

    # --------------------------------------------------------
    # Cross-validation
    # --------------------------------------------------------
    cv_results = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train_encoded,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    # --------------------------------------------------------
    # Save summary row
    # --------------------------------------------------------
    row = {
        "classifier": classifier_name,
        "accuracy_mean": cv_results["test_accuracy"].mean(),
        "accuracy_std": cv_results["test_accuracy"].std(),
        "precision_mean": cv_results["test_precision"].mean(),
        "precision_std": cv_results["test_precision"].std(),
        "recall_mean": cv_results["test_recall"].mean(),
        "recall_std": cv_results["test_recall"].std(),
        "f1_mean": cv_results["test_f1"].mean(),
        "f1_std": cv_results["test_f1"].std(),
        "roc_auc_mean": cv_results["test_roc_auc"].mean(),
        "roc_auc_std": cv_results["test_roc_auc"].std(),
    }

    for param_name, param_value in params.items():
        row[param_name] = str(param_value)

    results.append(row)

    model_elapsed = time.time() - model_start_time
    total_elapsed = time.time() - total_start_time

    print(f"  Completed in {model_elapsed:.2f} seconds")
    print(f"  Total elapsed: {total_elapsed:.2f} seconds\n")

print("Cross-validation complete.")
print(f"Total runtime: {time.time() - total_start_time:.2f} seconds")



Evaluating selected classifiers...

Number of classifiers: 2
Cross-validation folds: 5

[1/2] MLP
  Completed in 74.16 seconds
  Total elapsed: 74.16 seconds

[2/2] RBF SVM
  Completed in 77.24 seconds
  Total elapsed: 151.40 seconds

Cross-validation complete.
Total runtime: 151.40 seconds


In [7]:
# ============================================================
# Cell 6: Summarize and Rank Classifier Results
# ============================================================

df_cv_results = pd.DataFrame(results)

if df_cv_results.empty:
    raise ValueError("No cross-validation results found.")

# ------------------------------------------------------------
# Sort by primary metric (ROC-AUC)
# ------------------------------------------------------------
df_cv_results = df_cv_results.sort_values(
    by="roc_auc_mean",
    ascending=False
).reset_index(drop=True)

print("Classifier comparison (sorted by ROC-AUC):\n")

try:
    display(df_cv_results)
except:
    print(df_cv_results)

# ------------------------------------------------------------
# Display ranked summary
# ------------------------------------------------------------
print("\nRanked classifier summary:\n")

for i, row in df_cv_results.iterrows():
    print(
        f"{i+1}. {row['classifier']} | "
        f"ROC-AUC: {row['roc_auc_mean']:.4f} | "
        f"F1: {row['f1_mean']:.4f} | "
        f"Accuracy: {row['accuracy_mean']:.4f}"
    )

# ------------------------------------------------------------
# Identify top classifiers carried forward
# ------------------------------------------------------------
TOP_K_FINAL = min(2, len(df_cv_results))
top_rows = df_cv_results.head(TOP_K_FINAL)

print(f"\nTop {TOP_K_FINAL} classifiers retained for final training:\n")

for i, (_, row) in enumerate(top_rows.iterrows(), 1):
    print(f"{i}. {row['classifier']}")
    print(f"   ROC-AUC:  {row['roc_auc_mean']:.4f}")
    print(f"   F1 Score: {row['f1_mean']:.4f}")
    print(f"   Accuracy: {row['accuracy_mean']:.4f}")
    print()



Classifier comparison (sorted by ROC-AUC):



,classifier,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,...,hidden_layer_sizes,alpha,learning_rate_init,batch_size,max_iter,random_state,C,gamma,kernel,probability
0,RBF SVM,0.798194,0.008415,0.798329,0.008319,0.798194,0.008415,0.798171,0.008434,0.879229,...,NaN,NaN,NaN,NaN,NaN,42,100,0.01,rbf,True
1,MLP,0.765556,0.007953,0.766643,0.007726,0.765556,0.007953,0.765313,0.008020,0.843940,...,"(64, 32)",0.0001,0.001,32,300,42,NaN,NaN,NaN,NaN



Ranked classifier summary:

1. RBF SVM | ROC-AUC: 0.8792 | F1: 0.7982 | Accuracy: 0.7982
2. MLP | ROC-AUC: 0.8439 | F1: 0.7653 | Accuracy: 0.7656

Top 2 classifiers retained for final training:

1. RBF SVM
   ROC-AUC:  0.8792
   F1 Score: 0.7982
   Accuracy: 0.7982

2. MLP
   ROC-AUC:  0.8439
   F1 Score: 0.7653
   Accuracy: 0.7656



In [8]:
# ============================================================
# Cell 7: Train Final Models on Full Training Set
# ============================================================

trained_models = {}

print("Training final models on full training dataset...\n")

for config in classifier_configs:
    classifier_name = config["classifier"]
    params = config["params"].copy()

    print(f"Training: {classifier_name}")

    # --------------------------------------------------------
    # Instantiate model
    # --------------------------------------------------------
    if classifier_name == "MLP":
        if "hidden_layer_sizes" in params:
            params["hidden_layer_sizes"] = tuple(params["hidden_layer_sizes"])
        params["max_iter"] = 300
        params["random_state"] = RANDOM_SEED
        model = MLPClassifier(**params)

    elif classifier_name == "RBF SVM":
        params["kernel"] = "rbf"
        params["probability"] = True
        params["random_state"] = RANDOM_SEED
        model = SVC(**params)

    else:
        raise ValueError(f"Unsupported classifier: {classifier_name}")

    # --------------------------------------------------------
    # Train model
    # --------------------------------------------------------
    model.fit(X_train, y_train_encoded)

    # Store trained model
    trained_models[classifier_name] = model

    print(f"  Completed: {classifier_name}\n")

print("All models trained successfully.")



Training final models on full training dataset...

Training: MLP
  Completed: MLP

Training: RBF SVM
  Completed: RBF SVM

All models trained successfully.


In [9]:
# ============================================================
# Cell 8: Save Trained Models
# ============================================================

saved_model_paths = {}

print("Saving trained models...\n")

for model_name, model in trained_models.items():

    if model_name == "MLP":
        model_path = FINAL_MLP_MODEL_PATH
    elif model_name == "RBF SVM":
        model_path = FINAL_RBF_SVM_MODEL_PATH
    else:
        raise ValueError(f"Unsupported model name: {model_name}")

    with open(model_path, "wb") as f:
        pickle.dump(model, f)

    saved_model_paths[model_name] = model_path

    print(f"Saved: {model_name}")
    print(f"  Path: {model_path}\n")

print("All models saved successfully.")


Saving trained models...

Saved: MLP
  Path: /content/dip-ai-image-detection/metadata/models/final_mlp_model.pkl

Saved: RBF SVM
  Path: /content/dip-ai-image-detection/metadata/models/final_rbf_svm_model.pkl

All models saved successfully.


In [10]:
# ============================================================
# Cell 9: Save Cross-Validation Summary and Model Configs
# ============================================================

print("Saving cross-validation summary and model configurations...\n")

# ------------------------------------------------------------
# Save cross-validation summary
# ------------------------------------------------------------
df_cv_results.to_csv(CV_RESULTS_CSV, index=False)

print("Cross-validation summary saved successfully.")
print(f"CSV: {CV_RESULTS_CSV}")

# ------------------------------------------------------------
# Save per-model configuration files
# ------------------------------------------------------------
for config in classifier_configs:
    classifier_name = config["classifier"]
    params = config["params"]

    matching_row = df_cv_results[
        df_cv_results["classifier"] == classifier_name
    ].iloc[0]

    model_config = {
        "classifier": classifier_name,
        "hyperparameters": params,
        "num_features": NUM_FEATURES,
        "k_folds": K_FOLDS,
        "train_samples": int(len(X_train)),
        "selected_metric": "roc_auc",
        "roc_auc_mean": float(matching_row["roc_auc_mean"]),
        "model_path": str(saved_model_paths[classifier_name]),
    }

    if classifier_name == "MLP":
        output_path = BEST_MLP_MODEL_CONFIG_PATH
    elif classifier_name == "RBF SVM":
        output_path = BEST_RBF_SVM_MODEL_CONFIG_PATH
    else:
        raise ValueError(f"Unsupported classifier: {classifier_name}")

    with open(output_path, "w") as f:
        json.dump(model_config, f, indent=4)

    print(f"Saved config for {classifier_name}:")
    print(f"  JSON: {output_path}\n")

print("All model configurations saved successfully.")



Saving cross-validation summary and model configurations...

Cross-validation summary saved successfully.
CSV: /content/dip-ai-image-detection/metadata/results/cross_validation_results.csv
Saved config for MLP:
  JSON: /content/dip-ai-image-detection/metadata/models/best_mlp_model_config.json

Saved config for RBF SVM:
  JSON: /content/dip-ai-image-detection/metadata/models/best_rbf_svm_model_config.json

All model configurations saved successfully.
